# Lab 04 — Feature Engineering for Cybersecurity

*AI Cybersecurity Masterclass · Lab 04 · AI60secLab*

**Companion video:** [▶ Feature Engineering for Catching Hackers](https://www.youtube.com/watch?v=_32ty41DpSY&list=PLG4l6Ssi9A00)

## Research question
Can defensible derived features change the quality of threat detection?

A valid feature must be available at prediction time and must not contain target information.

In [1]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

Project root: /home/claude/run/exec
Data folder : /home/claude/run/exec/data/raw


In [2]:
import pandas as pd

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import add_safe_derived_features, drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.evaluation import binary_metrics, binary_metrics_frame

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [3]:
baseline = random_forest_pipeline(X_train)
baseline.fit(X_train, y_train)

base_pred = baseline.predict(X_test)
base_score = baseline.predict_proba(X_test)[:, 1]

base_metrics = binary_metrics(y_test, base_pred, base_score)
display(binary_metrics_frame("Original features", base_metrics))

,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,,
Original features,0.870804,0.816324,0.987558,0.893814,0.272243,0.012442,0.980502


In [4]:
X_train_engineered = add_safe_derived_features(X_train)
X_test_engineered = add_safe_derived_features(X_test)

new_features = sorted(
    set(X_train_engineered.columns) - set(X_train.columns)
)

print("New features:", new_features)
if new_features:
    display(X_train_engineered[new_features].head())

New features: ['bytes_per_second_safe', 'src_dst_byte_ratio_safe', 'src_dst_packet_ratio_safe']


,bytes_per_second_safe,src_dst_byte_ratio_safe,src_dst_packet_ratio_safe
0,3539.706451,1.497110,1.400000
1,65775.969645,0.017494,0.384615
2,8348.068239,0.027679,0.529412
3,831.329836,0.815824,1.000000
4,1784.383309,1.988848,1.571429


In [5]:
engineered = random_forest_pipeline(X_train_engineered)
engineered.fit(X_train_engineered, y_train)

eng_pred = engineered.predict(X_test_engineered)
eng_score = engineered.predict_proba(X_test_engineered)[:, 1]

eng_metrics = binary_metrics(y_test, eng_pred, eng_score)

comparison = pd.concat([
    binary_metrics_frame("Original features", base_metrics),
    binary_metrics_frame("Engineered features", eng_metrics),
])

display(comparison.round(4))

,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate,roc_auc
model,,,,,,,
Original features,0.8708,0.8163,0.9876,0.8938,0.2722,0.0124,0.9805
Engineered features,0.8729,0.8198,0.9859,0.8952,0.2655,0.0141,0.9809


## Research task

Design one additional feature.

Document:

1. the security hypothesis;
2. the mathematical definition;
3. whether it exists at prediction time;
4. the before/after metrics;
5. one reason the feature might fail in another environment.